In [1]:
import numpy as np
import plotly.graph_objects as go

In [2]:
L = 1.0 # length [m]
t_final = 1.0 # final time [s]
alpha = 0.1 # diffusivity [m^2/s]
POSITIONS = (0.1* L, 0.25*L, 0.5*L, 0.65*L, 0.85*L)

## 3.1 

In [3]:
def temperature(x, t):
    """Closed form solution for the one-dimensional heat equation."""
    return np.sin(np.pi * np.asarray(x) / L) * np.exp(
        -alpha * (np.pi / L) ** 2 * np.asarray(t)
    )

In [4]:
times = np.linspace(0.0, t_final, 201)
positions = np.linspace(0.0, L, 201)

trajectories = go.Figure()
styles = ("solid", "solid", "solid", "dash", "dash")
for x, style in zip(POSITIONS, styles):
    trajectories.add_trace(
        go.Scatter(
            x=times,
            y=temperature(x, times),
            mode="lines",
            name=f"x = {x:g}",
            line={"dash": style},
        )
    )
trajectories.update_layout(
    title="Temperature at five fixed positions",
    xaxis={"title": "Time t", "range": [0, t_final]},
    yaxis={"title": "Temperature u(x, t)"},
    legend_title="Rod position",
    )

In [5]:
values = temperature(positions[:, None], times[None, :])
heatmap = go.Figure(
    data=go.Heatmap(
        x=times,
        y=positions,
        z=values,
        zmin=0,
        zmax=1,
        colorscale="Inferno",
        colorbar={"title": "Temperature u(x, t)"},
    )
)
heatmap.update_layout(
    title="Temperature in the rod",
    xaxis={"title": "Time t"},
    yaxis={"title": "Position x"},
)

## Task 3.3

In [6]:
import torch
import torch.nn as nn


class Net(nn.Module):
    def __init__(self):
        super(Net,self).__init__()
        activation = nn.Tanh()
        self.net = nn.Sequential(
            nn.Linear(2, 10),
            activation,
            nn.Linear(10, 20),
            activation,
            nn.Linear(20, 10),
            activation,
            nn.Linear(10, 1),
        )

    def forward(self, tensor):
        return self.net(tensor)


In [7]:
def generate_training_points(n, L, T):
    x_interior = torch.rand(n, 1)*L
    x_interior.requires_grad_(True)
    t_interior = torch.rand(n, 1)*T
    t_interior.requires_grad_(True)

    t_boundary = torch.rand(n, 1)*T
    x_left = torch.zeros_like(t_boundary)
    x_right = torch.full_like(t_boundary, L)

    target_boundary = torch.zeros_like(t_boundary)

    x_initial = torch.rand(n,1)*L
    t_initial = torch.zeros_like(x_initial)

    target_initial = torch.sin(torch.pi * x_initial / L)
    
    return {"x_interior": x_interior, "t_interior": t_interior, "x_left": x_left, "x_right": x_right, "t_boundary": t_boundary, "x_initial": x_initial, "t_initial": t_initial, "target_boundary": target_boundary, "target_initial": target_initial}

In [8]:
def residual(model, x,t,alpha):
    u = model(torch.cat([x, t], dim=1))  
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]

    return u_t - alpha *u_xx


In [9]:
def compute_losses(model, pts, alpha):
    mse = nn.MSELoss()
    res = residual(model, pts["x_interior"], pts["t_interior"], alpha)
    loss_res = mse(res, torch.zeros_like(res))

    u_left = model(torch.cat([pts["x_left"], pts["t_boundary"]], dim=1))
    u_right = model(torch.cat([pts["x_right"], pts["t_boundary"]], dim=1))

    loss_boundary = mse(u_left, pts["target_boundary"]) + mse(u_right, pts["target_boundary"])

    u_initial =  model(torch.cat([pts["x_initial"], pts["t_initial"]], dim=1))
    loss_initial = mse(u_initial, pts["target_initial"])

    return loss_res, loss_boundary, loss_initial

In [10]:
# L2 error vs closed form

xs = torch.linspace(0, L, 101)
ts = torch.linspace(0, t_final, 101)
X, Tt = torch.meshgrid(xs, ts, indexing="ij")
grid = torch.stack([X.flatten(), Tt.flatten()], dim=1)

def temperature_torch(x, t):
    return torch.sin(torch.pi * x / L) * torch.exp(-alpha * (torch.pi / L) ** 2 * t)

u_true = temperature_torch(X, Tt).flatten()

def rel_l2(model):
    with torch.no_grad():
        u_pred = model(grid).flatten()
    return (torch.linalg.norm(u_pred - u_true) / torch.linalg.norm(u_true)).item()


In [11]:
alpha = 0.1
L = 1.0
T = 1.0
model = Net()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

lambda_res, lambda_boundary, lambda_initial = 1.0, 1.0, 1.0

history = {"total": [], "res": [], "boundary": [], "initial": []}
l2_epochs, l2_values = [], []

    
for epoch in range(5000):
    pts = generate_training_points(1000, L, T)  
    optimizer.zero_grad()
    loss_res, loss_boundary, loss_initial = compute_losses(model, pts, alpha)
    loss = lambda_res * loss_res + lambda_boundary * loss_boundary + lambda_initial * loss_initial
    loss.backward()
    optimizer.step()

    history["total"].append(loss.item())
    history["res"].append(loss_res.item())
    history["boundary"].append(loss_boundary.item())
    history["initial"].append(loss_initial.item())

    if epoch % 50 == 0:
        l2_epochs.append(epoch + 1)
        l2_values.append(rel_l2(model))

    if epoch % 500 == 0:
        print(f"{epoch + 1}: total={loss.item():.2e} res={loss_res.item():.2e} "
              f"boundary={loss_boundary.item():.2e} initial={loss_initial.item():.2e}")

final_l2 = rel_l2(model)
if not l2_epochs or l2_epochs[-1] != epoch + 1:
    l2_epochs.append(epoch + 1)
    l2_values.append(final_l2)
print(f"Final relative L2 error: {final_l2:.4e} ({100 * final_l2:.3f}%)")


1: total=4.56e-01 res=2.77e-03 boundary=1.07e-02 initial=4.42e-01
501: total=3.48e-03 res=2.38e-03 boundary=7.42e-04 initial=3.63e-04
1001: total=8.86e-04 res=5.32e-04 boundary=2.71e-04 initial=8.26e-05
1501: total=6.35e-04 res=3.22e-04 boundary=2.32e-04 initial=8.16e-05
2001: total=4.16e-04 res=1.76e-04 boundary=1.86e-04 initial=5.35e-05
2501: total=3.32e-04 res=1.42e-04 boundary=1.37e-04 initial=5.25e-05
3001: total=2.53e-04 res=1.14e-04 boundary=1.13e-04 initial=2.59e-05
3501: total=1.92e-04 res=9.50e-05 boundary=7.66e-05 initial=2.06e-05
4001: total=1.53e-04 res=6.99e-05 boundary=6.38e-05 initial=1.97e-05
4501: total=1.24e-04 res=7.10e-05 boundary=4.20e-05 initial=1.09e-05
Final relative L2 error: 5.4915e-03 (0.549%)


In [17]:
# Training losses: one value per parameter update.
loss_figure = go.Figure()
for key, values in history.items():
    loss_figure.add_trace(go.Scatter(
        x=list(range(1, len(values) + 1)), y=values,
        mode="lines", name=key,
    ))
loss_figure.update_layout(
    title="PINN training losses",
    xaxis_title="Epoch", yaxis_title="Loss",
    yaxis_type="log", template="plotly_white",
)
loss_figure.show()

# Relative error on the separate evaluation grid.
l2_figure = go.Figure(go.Scatter(
    x=l2_epochs, y=l2_values, mode="lines+markers", name="Relative L2 error",
))
l2_figure.update_layout(
    title="PINN error against the closed-form solution",
    xaxis_title="Epoch", yaxis_title="Relative L2 error",
    yaxis_type="log", template="plotly_white",
)
l2_figure.show()
loss_figure.write_image("pinn_training_losses.png")
l2_figure.write_image("pinn_relative_error.png")

In [15]:
# Compare at the same five positions as Task 3.1.
comparison_times = torch.linspace(0, T, 201).reshape(-1, 1)
comparison_figure = go.Figure()
colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"]
for x0, color in zip(POSITIONS, colors):
    input_pairs = torch.cat([
        torch.full_like(comparison_times, x0), comparison_times,
    ], dim=1)
    with torch.no_grad():
        u_pred = model(input_pairs).flatten().tolist()
    u_exact = temperature_torch(
        torch.full_like(comparison_times, x0), comparison_times,
    ).flatten().tolist()
    for label, values, dash in [("Exact", u_exact, "solid"), ("PINN", u_pred, "dash")]:
        comparison_figure.add_trace(go.Scatter(
            x=comparison_times.flatten().tolist(), y=values,
            mode="lines", name=f"{label}: x={x0:g}", legendgroup=str(x0),
            line={"color": color, "dash": dash},
        ))
comparison_figure.update_layout(
    title="PINN and exact temperature at five rod positions",
    xaxis_title="Time t", yaxis_title="Temperature u(x, t)",
    template="plotly_white",
)
comparison_figure.show()
comparison_figure.write_image("pinn_comparison.png")
